# TorchServe

PyTorch's official model serving framework for production deployment.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**TorchServe** is the official model serving library from PyTorch, designed to make deploying PyTorch models at scale easy and production-ready.

### What is it?

TorchServe is a flexible serving framework that:
- Is the **official PyTorch** serving solution (from AWS and Facebook)
- Uses **Model Archive (MAR)** format for packaging models
- Provides **REST and gRPC APIs** out of the box
- Supports **custom handlers** for preprocessing/postprocessing
- Enables **multi-model serving** on single instance
- Includes **A/B testing** and **model versioning**
- Works with **TorchScript**, **eager mode**, and **ONNX**

### Why use it?

Key benefits:
- **PyTorch Native**: Best integration with PyTorch ecosystem
- **Production Ready**: Built for enterprise deployment
- **Flexible Handlers**: Custom logic for any use case
- **Easy Deployment**: From development to production seamlessly
- **AWS Integration**: First-class support on SageMaker
- **Active Development**: Backed by AWS and Meta
- **Model Management**: Register, version, and manage models dynamically

### When to use it?

TorchServe is ideal when:
- Your models are built with **PyTorch**
- Need **custom preprocessing/postprocessing** logic
- Want **official PyTorch support** and long-term maintenance
- Deploying on **AWS infrastructure** (especially SageMaker)
- Need **multi-model serving** with dynamic loading
- Require **enterprise features** (metrics, logging, management API)

**Trade-off**: More complex than simple REST wrappers, but much more feature-rich for production.

## Key Features

### Core Capabilities of TorchServe

| Feature | Description | Benefit |
|---------|-------------|----------|
| **MAR Format** | Model Archive for packaging models | Portable, versioned deployments |
| **Custom Handlers** | Python-based pre/post processing | Any business logic |
| **Multi-Model** | Serve multiple models on one instance | Resource efficiency |
| **Dynamic Loading** | Load/unload models via API | Zero-downtime updates |
| **Batch Inference** | Automatic request batching | Higher throughput |
| **A/B Testing** | Traffic splitting between versions | Safe model updates |
| **RESTful API** | HTTP endpoints for inference | Easy integration |
| **gRPC Support** | High-performance RPC protocol | Low latency |
| **Model Versioning** | Serve multiple versions simultaneously | Gradual rollouts |
| **Metrics API** | Prometheus-compatible metrics | Observability |
| **Snapshot** | Save/restore server state | Quick recovery |
| **PyTorch Ecosystem** | TorchScript, ONNX, eager mode | Flexibility |

## Architecture Overview

TorchServe uses a multi-process architecture for isolation and scalability:

```
┌─────────────────────────────────────────────────────────┐
│                   CLIENT LAYER                          │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐            │
│  │   HTTP   │  │   gRPC   │  │Management│            │
│  │  Client  │  │  Client  │  │   API    │            │
│  └────┬─────┘  └────┬─────┘  └────┬─────┘            │
└───────┼─────────────┼─────────────┼───────────────────┘
        │             │             │
┌───────▼─────────────▼─────────────▼───────────────────┐
│              TORCHSERVE FRONTEND                       │
│  ┌──────────────────────────────────────────────────┐ │
│  │         Inference API (Port 8080)                │ │
│  │  • /predictions/{model_name}                     │ │
│  │  • /explanations/{model_name}                    │ │
│  └──────────────────────────────────────────────────┘ │
│  ┌──────────────────────────────────────────────────┐ │
│  │       Management API (Port 8081)                 │ │
│  │  • /models (register/unregister)                 │ │
│  │  • /models/{model_name} (scale workers)          │ │
│  └──────────────────────────────────────────────────┘ │
│  ┌──────────────────────────────────────────────────┐ │
│  │         Metrics API (Port 8082)                  │ │
│  │  • Prometheus metrics                            │ │
│  └──────────────────────────────────────────────────┘ │
└─────────────────────┬──────────────────────────────────┘
                      │
┌─────────────────────▼──────────────────────────────────┐
│             TORCHSERVE BACKEND                         │
│                                                        │
│  ┌────────────────────────────────────────────────┐   │
│  │          Model Manager                         │   │
│  │  • Load/unload models                          │   │
│  │  • Worker pool management                      │   │
│  │  • Request routing                             │   │
│  └─────────────────┬──────────────────────────────┘   │
│                    │                                   │
│   ┌────────────────┼────────────────┐                 │
│   │                │                │                 │
│   ▼                ▼                ▼                 │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐           │
│  │ Model    │  │ Model    │  │ Model    │           │
│  │ Worker 1 │  │ Worker 2 │  │ Worker 3 │  ...      │
│  └──────────┘  └──────────┘  └──────────┘           │
│                                                        │
│  Each worker:                                         │
│  ┌────────────────────────────────────────────────┐   │
│  │  1. Load model (TorchScript/eager/ONNX)       │   │
│  │  2. Run custom handler                        │   │
│  │  3. Pre-process → Inference → Post-process    │   │
│  │  4. Return result                             │   │
│  └────────────────────────────────────────────────┘   │
└────────────────────────────────────────────────────────┘
```

### Key Components

1. **Frontend**: Handles HTTP/gRPC requests, routing
2. **Backend**: Manages model workers and execution
3. **Model Workers**: Isolated Python processes running models
4. **Custom Handlers**: User-defined logic for pre/post processing
5. **MAR Files**: Packaged models with dependencies and configs

## Installation

### Prerequisites

- **Python 3.8-3.11**
- **PyTorch 1.12+**
- **Java 11+** (for TorchServe runtime)

### Installation via pip

In [ ]:
# Install TorchServe
# pip install torchserve torch-model-archiver torch-workflow-archiver

# Install PyTorch if not already installed
# pip install torch torchvision

import sys
print(f"Python version: {sys.version}")

# Verify installation
# torchserve --version

### Installation via Docker (Recommended for Production)

In [ ]:
# Pull official TorchServe image
docker_install = '''
# CPU version
docker pull pytorch/torchserve:latest

# GPU version
docker pull pytorch/torchserve:latest-gpu

# Run TorchServe
docker run --rm -it \\
  -p 8080:8080 \\
  -p 8081:8081 \\
  -p 8082:8082 \\
  pytorch/torchserve:latest
'''

print("Docker installation:")
print(docker_install)

## Basic Usage

### Step 1: Create and Export a PyTorch Model

In [ ]:
# Create a simple PyTorch model
model_creation = '''
import torch
import torch.nn as nn

# Define model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)
    
    def forward(self, x):
        return self.fc(x)

# Create and save model
model = SimpleModel()
model.eval()

# Save as TorchScript
scripted_model = torch.jit.script(model)
torch.jit.save(scripted_model, "model.pt")

# Or save as state dict (requires custom handler)
# torch.save(model.state_dict(), "model.pth")
'''

print("Model creation and export:")
print(model_creation)

### Step 2: Package Model into MAR (Model Archive)

In [ ]:
# Create MAR file using torch-model-archiver
mar_creation = '''
# Basic MAR creation
torch-model-archiver \\
  --model-name my_model \\
  --version 1.0 \\
  --serialized-file model.pt \\
  --handler image_classifier \\
  --export-path model_store/

# With custom handler
torch-model-archiver \\
  --model-name my_model \\
  --version 1.0 \\
  --model-file model.py \\
  --serialized-file model.pth \\
  --handler custom_handler.py \\
  --extra-files index_to_name.json \\
  --export-path model_store/

# Built-in handlers:
# - image_classifier
# - image_segmenter
# - object_detector
# - text_classifier
'''

print("MAR creation:")
print(mar_creation)

### Step 3: Start TorchServe

In [ ]:
# Start TorchServe server
start_server = '''
# Basic start
torchserve \\
  --start \\
  --model-store model_store/ \\
  --models my_model=my_model.mar

# With configuration
torchserve \\
  --start \\
  --model-store model_store/ \\
  --models my_model=my_model.mar \\
  --ts-config config.properties

# Stop server
# torchserve --stop
'''

print("Start TorchServe:")
print(start_server)
print("\nPorts:")
print("8080: Inference API")
print("8081: Management API")
print("8082: Metrics API")

### Step 4: Make Inference Requests

In [ ]:
# Make inference requests
inference_example = '''
import requests

# Inference request
response = requests.post(
    'http://localhost:8080/predictions/my_model',
    files={'data': open('input.jpg', 'rb')}
)

print(response.json())

# Or with JSON input
import json
response = requests.post(
    'http://localhost:8080/predictions/my_model',
    json={"data": [1.0, 2.0, 3.0]}
)
'''

print("Inference example:")
print(inference_example)

## Advanced Features

### 1. Custom Handlers

In [ ]:
# Custom handler with pre/post processing
custom_handler = '''
# custom_handler.py
from ts.torch_handler.base_handler import BaseHandler
import torch
import json

class CustomHandler(BaseHandler):
    def __init__(self):
        super().__init__()
        self.initialized = False
    
    def initialize(self, context):
        """Load model and initialize."""
        self.manifest = context.manifest
        properties = context.system_properties
        model_dir = properties.get("model_dir")
        
        # Load model
        serialized_file = self.manifest["model"]["serializedFile"]
        model_pt_path = os.path.join(model_dir, serialized_file)
        self.model = torch.jit.load(model_pt_path)
        self.model.eval()
        
        # Load any additional files
        self.mapping = json.load(open(os.path.join(model_dir, "index_to_name.json")))
        
        self.initialized = True
    
    def preprocess(self, data):
        """Transform raw input into tensor."""
        # Custom preprocessing logic
        processed_data = []
        for row in data:
            # Extract data from request
            input_data = row.get("data") or row.get("body")
            
            # Convert to tensor
            tensor = torch.FloatTensor(input_data)
            processed_data.append(tensor)
        
        return torch.stack(processed_data)
    
    def inference(self, data):
        """Run model inference."""
        with torch.no_grad():
            results = self.model(data)
        return results
    
    def postprocess(self, inference_output):
        """Transform model output into response."""
        # Get predictions
        predictions = inference_output.argmax(dim=1).tolist()
        
        # Map to class names
        results = [self.mapping[str(pred)] for pred in predictions]
        
        return results

_service = CustomHandler()

def handle(data, context):
    if not _service.initialized:
        _service.initialize(context)
    
    if data is None:
        return None
    
    data = _service.preprocess(data)
    data = _service.inference(data)
    data = _service.postprocess(data)
    
    return data
'''

print("Custom handler example:")
print(custom_handler)

### 2. Dynamic Model Management

In [ ]:
# Dynamic model registration and scaling
management_api = '''
import requests

# Register a model
requests.post(
    'http://localhost:8081/models',
    params={
        'url': 'my_model.mar',
        'initial_workers': 1,
        'synchronous': True
    }
)

# Scale workers
requests.put(
    'http://localhost:8081/models/my_model',
    params={'min_worker': 2, 'max_worker': 4}
)

# Set default version
requests.put(
    'http://localhost:8081/models/my_model/1.0/set-default'
)

# Unregister model
requests.delete('http://localhost:8081/models/my_model')

# List models
response = requests.get('http://localhost:8081/models')
print(response.json())
'''

print("Management API examples:")
print(management_api)

### 3. Batch Inference

In [ ]:
# Configure batch inference
batch_config = '''
# In config.properties
inference_address=http://0.0.0.0:8080
management_address=http://0.0.0.0:8081
metrics_address=http://0.0.0.0:8082

# Batch settings
default_workers_per_model=2
job_queue_size=100

# Batch configuration (per model)
batch_size=8              # Max batch size
max_batch_delay=100       # Max delay in ms

# Or via MAR config
# In model-config.yaml:
minWorkers: 1
maxWorkers: 4
batchSize: 8
maxBatchDelay: 100
responseTimeout: 120
'''

print("Batch inference configuration:")
print(batch_config)

### 4. Model Versioning and A/B Testing

In [ ]:
# Serve multiple versions
versioning = '''
# Register version 1.0
requests.post(
    'http://localhost:8081/models',
    params={'url': 'my_model_v1.mar', 'model_name': 'my_model'}
)

# Register version 2.0
requests.post(
    'http://localhost:8081/models',
    params={'url': 'my_model_v2.mar', 'model_name': 'my_model'}
)

# Request specific version
# Default version
requests.post('http://localhost:8080/predictions/my_model', ...)

# Specific version
requests.post('http://localhost:8080/predictions/my_model/1.0', ...)
requests.post('http://localhost:8080/predictions/my_model/2.0', ...)

# A/B testing: Use load balancer to split traffic
# 90% to v1.0, 10% to v2.0
'''

print("Model versioning:")
print(versioning)

## Use Cases

### Real-World Applications

#### Use Case 1: Image Classification Service

In [ ]:
# Production image classification
image_service = '''
# Export ResNet model
import torch
import torchvision.models as models

model = models.resnet50(pretrained=True)
model.eval()
scripted_model = torch.jit.script(model)
torch.jit.save(scripted_model, "resnet50.pt")

# Create MAR
torch-model-archiver \\
  --model-name resnet50 \\
  --version 1.0 \\
  --serialized-file resnet50.pt \\
  --handler image_classifier \\
  --extra-files index_to_name.json \\
  --export-path model_store/

# Start server
torchserve --start \\
  --model-store model_store/ \\
  --models resnet50=resnet50.mar \\
  --ncs

# Client usage
import requests
response = requests.post(
    'http://localhost:8080/predictions/resnet50',
    files={'data': open('cat.jpg', 'rb')}
)
print(response.json())  # [{"tiger_cat": 0.8, ...}]
'''

print("Image classification service:")
print(image_service)

## Best Practices

### Recommended Practices

#### 1. Model Packaging

- **Use TorchScript**: `torch.jit.script()` for better performance
- **Include dependencies**: Add extra files needed by handler
- **Version models**: Always specify version in MAR
- **Test locally**: Validate MAR before production

#### 2. Handler Design

- Inherit from `BaseHandler` for structure
- Keep preprocessing lightweight
- Use `@torch.no_grad()` in inference
- Handle errors gracefully
- Log important operations

#### 3. Performance

- Enable batch inference for throughput
- Scale workers based on load
- Use appropriate `batch_size` and `max_batch_delay`
- Monitor queue lengths
- Use GPU workers when available

#### 4. Production Deployment

- Use Docker for consistency
- Set resource limits (CPU/memory)
- Enable metrics endpoint
- Implement health checks
- Use load balancers for HA

#### 5. Model Updates

- Register new versions dynamically
- Test on subset of traffic first
- Keep previous version available
- Monitor metrics after updates

## Common Pitfalls

### What to Avoid

#### 1. Java Not Installed

**Problem**: TorchServe requires Java 11+

**Symptom**: `Java not found` error

**Solution**: Install Java 11 or newer

#### 2. Handler Import Errors

**Problem**: Custom handler can't import dependencies

**Solution**: Include all Python files in MAR with `--extra-files`

#### 3. Model Not Loading

**Problem**: Model fails to load in worker

**Symptom**: Worker crashes, "Model not found"

**Solution**: Check MAR contents, verify paths in handler

#### 4. Memory Issues

**Problem**: Workers consuming too much memory

**Solution**: Reduce `max_workers`, optimize model, use smaller batches

#### 5. Slow Inference

**Problem**: High latency despite GPU

**Solution**: Enable batching, increase workers, profile handler

## Production Deployment

### Docker Deployment

In [ ]:
# Production Dockerfile
dockerfile = '''
FROM pytorch/torchserve:latest-gpu

# Copy MAR files
COPY model_store/ /home/model-server/model-store/

# Copy config
COPY config.properties /home/model-server/config.properties

# Expose ports
EXPOSE 8080 8081 8082

# Health check
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
  CMD curl -f http://localhost:8080/ping || exit 1

# Start TorchServe
CMD ["torchserve", \\
     "--start", \\
     "--model-store", "/home/model-server/model-store", \\
     "--ts-config", "/home/model-server/config.properties", \\
     "--models", "all"]
'''

print("Production Dockerfile:")
print(dockerfile)

## Monitoring and Observability

In [ ]:
# TorchServe metrics
metrics_guide = '''
# Metrics endpoint
http://localhost:8082/metrics

# Key metrics:
# Requests
ts_inference_requests_total{model_name="my_model",model_version="1.0"}
ts_inference_latency_microseconds{model_name="my_model"}
ts_queue_latency_microseconds{model_name="my_model"}

# Workers
ts_worker_thread_count{model_name="my_model"}
ts_worker_memory_used{model_name="my_model"}

# System
ts_queue_size{model_name="my_model"}
cpu_utilization
memory_used
disk_utilization

# Custom metrics in handler
self.context.metrics.add_metric(
    "CustomMetric", 
    value,
    unit="count",
    dimension_names=["model"],
    dimension_values=["my_model"]
)
'''

print("Metrics guide:")
print(metrics_guide)

## Comparison with Alternatives

### How TorchServe Compares

| Feature | TorchServe | Triton | TF Serving | BentoML |
|---------|------------|--------|------------|----------|
| **PyTorch Focus** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐ | ⭐⭐⭐⭐ |
| **Multi-Framework** | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐ | ⭐⭐⭐⭐⭐ |
| **Ease of Use** | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Custom Logic** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Performance** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **AWS Integration** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **Management API** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐ |

### When to Choose TorchServe

**Choose TorchServe when:**
- ✅ Using **PyTorch** models exclusively
- ✅ Need **custom preprocessing** logic
- ✅ Want **official PyTorch** support
- ✅ Deploying on **AWS/SageMaker**
- ✅ Need **dynamic model management**
- ✅ Require **MAR packaging** for portability

**Choose alternatives when:**
- ❌ Using multiple frameworks (use Triton)
- ❌ Need absolute peak performance (use Triton/TensorRT)
- ❌ Want Python-first API (use BentoML)
- ❌ Only TensorFlow models (use TF Serving)

## Resources

### Official Documentation

- **GitHub**: https://github.com/pytorch/serve
- **Documentation**: https://pytorch.org/serve/
- **Examples**: https://github.com/pytorch/serve/tree/master/examples
- **Model Zoo**: https://github.com/pytorch/serve/blob/master/docs/model_zoo.md

### Tutorials

- **Quick Start**: https://pytorch.org/serve/getting_started.html
- **Custom Handlers**: https://pytorch.org/serve/custom_service.html
- **Batch Inference**: https://pytorch.org/serve/batch_inference_with_ts.html

### Community

- **PyTorch Forums**: https://discuss.pytorch.org/c/torchserve/
- **GitHub Discussions**: https://github.com/pytorch/serve/discussions
- **Stack Overflow**: Tag `torchserve`

### Related Tools

- **torch-model-archiver**: MAR creation tool
- **torch-workflow-archiver**: Workflow packaging
- **AWS SageMaker**: Managed TorchServe hosting